# 4. Quality audit and limitations

<!-- Drafted with the help of an AI assistant (Claude, Anthropic), disclosed per course rules. -->

How correct is the output? Three checks, from the cheapest to the most expensive: integrity of the tables (done on the whole output), agreement of the tree with the source (done on a sample only) and a reading of random papers (not done by a person yet). The page says which is which.

## 4.1. Integrity of the tables: whole output

The original parse output and the derived output were checked separately, without rerunning the parser. For each of 200 shards, the filter verified the original hashes, removed the empty document roots, and checked the derived tables before publication. The original QA expects a root for each non-missing version; derived QA expects no hierarchy nodes for `empty` versions and a root for the remaining represented versions. Both check schemas, IDs, ownership, parent links and reachability.


In [ ]:
import json
from pathlib import Path

import pandas as pd

DATA = Path("../data/m2")
check = json.loads((DATA / "check_tables.json").read_text())
status = json.loads((DATA / "qa_status.json").read_text())
papers = pd.read_csv(DATA / "m2_papers.csv", dtype={"arxiv_id": str})
versions = pd.read_csv(DATA / "m2_versions.csv", dtype={"arxiv_id": str})
findings = pd.read_csv(DATA / "audit_findings.csv")
flags = pd.read_csv(DATA / "sentence_flags.csv")

In [ ]:
pd.DataFrame(
    {
        "Value": {
            "Problems found": check["n_problems"],
            "Publications": f"{check['papers']:,}",
            "Versions": f"{check['versions']:,}",
            "Rows of elements": f"{check['rows']['elements']:,}",
            "Rows of hierarchy": f"{check['rows']['hierarchy']:,}",
            "Rows of bibtex": f"{check['rows']['bibtex']:,}",
        }
    }
)

In [ ]:
kinds = pd.Series(check["elements_by_kind"]).sort_values(ascending=False)
kinds.rename("elements").to_frame().assign(share_pct=(100 * kinds / kinds.sum()).round(2))

In [ ]:
roots = int((versions["status"] != "missing").sum())
pd.Series(
    {
        "elements in the tables = elements in m2_papers.csv": check["rows"]["elements"] == int(papers["elements"].sum()),
        "document roots = versions that have a folder": check["elements_by_kind"]["document"] == roots,
        "hierarchy rows = m2_papers.csv": check["rows"]["hierarchy"] == int(papers["hierarchy"].sum()),
        "bibtex rows = m2_papers.csv": check["rows"]["bibtex"] == int(papers["bibtex"].sum()),
    },
    name="agrees",
).to_frame()

`qa_status.json` of the same QA run records what was and was not done:

In [ ]:
pd.Series(status, name='status').to_frame()

### Derived output after filtering empty roots

The version journal still records **26,491 versions**. Derived QA counts only versions represented in the hierarchy: 25,041 (25,032 parsed and 9 `no_root`). The 1,450 `empty` versions remain in the journal with no elements or hierarchy edges. Source comparison in the next section was performed before the filter on parsed versions; the filter changes only empty roots.


In [ ]:
derived = json.loads((DATA / "filtered_qa.json").read_text())
filtered = json.loads((DATA / "filter_report.json").read_text())
pd.DataFrame({"Original": [check["rows"][k] for k in ("elements", "hierarchy", "bibtex")], "Filtered": [derived["rows"][k] for k in ("elements", "hierarchy", "bibtex")]}, index=["Elements", "Hierarchy", "BibTeX"])


In [ ]:
assert filtered["removed_roots"] == int((versions["status"] == "empty").sum())
assert derived["ok"] and derived["n_problems"] == 0
print(f"Removed {filtered['removed_roots']:,} empty roots from {filtered['shards']} shards; derived QA passed.")


## 4.2. Agreement with the source and reading of papers: sample only

A subsequent audit `qa-thien-01` compared the official output with source for **2,407 parsed versions from 1,079 papers** (1,114 papers have source available) from the multi-Thiên run; this is a selected subset, not the whole corpus. See [Parsing statistics](02-data-stats.ipynb). An earlier comparison and reading used a separate **sample of 100 publications** (121 versions) parsed with a previous revision of the code. The reading used 30 random publications (seed 2026) and was done with the help of an AI assistant; **the score sheet is still to be filled by a team member**. The table lists what the reading and an automatic scan of the sentences found on that sample, not on the official output.

In [ ]:
findings.set_index('category')

In [ ]:
flags.set_index('scope')

## 4.3. Empty TeX files and versions without a root

M1 QA flagged **69 files** that were empty or whitespace-only, across 48 versions of 35 publications. All 48 versions have M2 status `parsed`; none is among the nine `no_root` versions. We re-downloaded those 48 exact versions and used the parser's gather rules to inspect reachability. The selected root matched the recorded M2 root in every case.

For 27 versions (42 warnings), all downloaded TeX files matched the available M1 checkpoint hashes: **40 warning paths were reachable** and two were not. For the remaining 21 versions (27 warnings), the checkpoint index was unavailable locally; all 27 paths were reachable in the re-downloaded sources, but their byte identity with the M1 inputs remains unverified. Reachability of an empty file does not establish whether content was intended, omitted, or deliberately left empty.


In [ ]:
audit_rows = [json.loads(line) for line in (DATA / "tex_warning_reachability.jsonl").read_text(encoding="utf8").splitlines()]
comparison = []
for verified, label in [(True, "Matched M1 hashes"), (False, "M1 hashes not verified")]:
    group = [row for row in audit_rows if row["hash_verified"] is verified]
    comparison.append({"Evidence": label, "Versions": len(group), "Warning files": sum(len(row["warning_paths"]) for row in group), "Reachable files": sum(len(row["reachable_warnings"]) for row in group)})
pd.DataFrame(comparison).set_index("Evidence")


The nine `no_root` versions belong to seven papers. Their re-downloaded `.tex` files were nonempty and did not contain a literal `\documentclass`. All TeX hashes matched the local M1 index for five versions; the other four were checked by file name and size but not by hash. Plain-TeX-style markers were observed in some files; this lexical inspection is not a compilation test. These versions remain `no_root`, separate from `no_tex` and `unavailable`.


## 4.4. Known limitations

- **Author macros are not expanded.** A sentence such as `\toolname achieves ...` keeps the macro name; `\newcommand` definitions are dropped, not applied.
- **Template front matter.** The commands of the common journal templates are dropped, but a template the parser has not seen can leave its author block or its notes among the sentences of the first section.
- **References are heuristics.** `citation_key`, `year`, `doi` and `url` are reliable; `authors`, `title`, `journal` and `entry_type` depend on the style of a hand-written `\bibitem`. A share of the parsed versions has no reference at all (see [Parsing statistics](02-data-stats.ipynb)): a `.bbl` that could not be recovered or parsed, or a `.bib` the corpus does not hold.
- **A few parsed versions hold almost nothing** (fewer than 20 elements): a stub root file or an unrecognised structure. They are listed in the statistics and not corrected.
- **Equal elements under one parent are stored once**, so a sentence repeated inside the same section appears once in `hierarchy`, and the order of siblings is the order of the rows, not a column.
- **Not followed:** `\subfile`, `\import` and `\InputIfFileExists`.
- **Source comparison is limited to a selected subset**, and the historical 30-paper reading comes from a separate 100-paper sample; the human score sheet remains unfilled.